
## 1. Importações e configurações

Nesta seção, são importados os módulos nativos do Python
necessários para o desenvolvimento da análise financeira.

- **csv:** leitura e processamento das transações do arquivo CSV.
- **json:** exportação dos resultados para um arquivo JSON.
- **datetime:** validação, conversão e manipulação das datas.

Também é definida a constante `LIMITE_SUSPEITO`,
utilizada para identificar transações com valores
superiores a R$ 10.000,00.


In [109]:
import csv
import json
from datetime import datetime

LIMITE_SUSPEITO = 10000.00


## 2. Leitura do arquivo CSV

Nesta etapa, será implementada a leitura do arquivo
de transações utilizando o módulo nativo csv do Python.

A função será responsável por carregar os registros
e tratar possíveis erros de arquivo inexistente.


In [110]:

def ler_transacoes(caminho_arquivo):
    try:
        with open(caminho_arquivo, "r", encoding="utf-8-sig", newline="") as arquivo:
            leitor = csv.DictReader(arquivo)
            transacoes = list(leitor)

        return transacoes

    except FileNotFoundError:
        print(f"Erro: arquivo '{caminho_arquivo}' não encontrado.")
        return []


In [111]:

transacoes = ler_transacoes("transacoes.csv")

print(f"Total de transações lidas: {len(transacoes)}")

for transacao in transacoes[:3]:
    print(transacao)


Total de transações lidas: 20
{'id': '1', 'data': '2026-01-05', 'cliente_id': 'CLI001', 'tipo': 'credito', 'valor': '3500.00', 'descricao': 'Salario janeiro', 'categoria': 'salario'}
{'id': '2', 'data': '2026-01-12', 'cliente_id': 'CLI002', 'tipo': 'debito', 'valor': '180.50', 'descricao': 'Supermercado', 'categoria': 'compra'}
{'id': '3', 'data': '2026-01-15', 'cliente_id': 'CLI003', 'tipo': 'credito', 'valor': '1200.00', 'descricao': 'Transferencia recebida', 'categoria': 'transferencia'}



## 3. Validação e limpeza dos dados

Nesta etapa, serão realizadas validações nos registros
carregados do arquivo CSV.

A função validar_transacao() será responsável por
verificar os campos obrigatórios, os tipos de dados
e os valores permitidos.

Os registros inválidos serão descartados sem
interromper a execução do programa.


In [112]:

def validar_transacao(transacao):

    # 1. Verificar campos obrigatórios
    campos = [
        "id", "data", "cliente_id",
        "tipo", "valor", "descricao", "categoria"
    ]

    for campo in campos:
        if not isinstance(transacao.get(campo), str):
            return None

        if not transacao[campo].strip():
            return None

    # 2. Validar o ID
    id_texto = transacao["id"].strip()

    if not id_texto.isascii() or not id_texto.isdigit():
        return None

    # 3. Validar o cliente
    cliente_id = transacao["cliente_id"].strip()

    # 4. Validar a data
    try:
        data_texto = transacao["data"].strip()
        data = datetime.strptime(data_texto, "%Y-%m-%d")

        if data.strftime("%Y-%m-%d") != data_texto:
            return None

    except ValueError:
        return None

    # 5. Validar o tipo
    tipo = transacao["tipo"].strip()

    if tipo not in ["credito", "debito"]:
        return None

    # 6. Validar o valor
    try:
        valor = float(transacao["valor"])

        if not 0 < valor < float("inf"):
            return None

    except ValueError:
        return None

    # 7. Retornar registro validado
    return {
        "id": int(id_texto),
        "data": data,
        "cliente_id": cliente_id,
        "tipo": tipo,
        "valor": valor,
        "descricao": transacao["descricao"].strip(),
        "categoria": transacao["categoria"].strip()
    }


In [114]:

validas = []
invalidas = []

for transacao in transacoes:
    resultado = validar_transacao(transacao)

    if resultado is not None:
        validas.append(resultado)
    else:
        invalidas.append(transacao)

print("===== RESUMO DA LIMPEZA =====")
print(f"Total de linhas lidas: {len(transacoes)}")
print(f"Linhas válidas: {len(validas)}")
print(f"Linhas inválidas: {len(invalidas)}")


===== RESUMO DA LIMPEZA =====
Total de linhas lidas: 20
Linhas válidas: 15
Linhas inválidas: 5


In [115]:

print("===== REGISTROS INVÁLIDOS =====")

for transacao in invalidas:
    print(
        f"ID: {transacao['id']} | "
        f"Cliente: {transacao['cliente_id']} | "
        f"Valor: {transacao['valor']}"
    )


===== REGISTROS INVÁLIDOS =====
ID: 16 | Cliente: CLI001 | Valor: abc
ID: 17 | Cliente:  | Valor: 500.00
ID: 18 | Cliente: CLI002 | Valor: -150.00
ID: 19 | Cliente: CLI003 | Valor: 800.00
ID: abc | Cliente: CLI001 | Valor: 100.00



## 4. Manipulação de datas

Nesta etapa, serão identificadas as datas das
transações mais antiga e mais recente.

Também será calculada a quantidade de dias
entre essas transações e extraído o mês
de cada registro para o agrupamento mensal.


In [116]:

# Extrair os meses das transações
for transacao in validas:
    transacao["mes"] = transacao["data"].strftime("%Y-%m")

# Identificar as datas extremas
data_mais_antiga = min(t["data"] for t in validas)
data_mais_recente = max(t["data"] for t in validas)

# Calcular a diferença de dias
dias_periodo = (data_mais_recente - data_mais_antiga).days

print("===== PERÍODO ANALISADO =====")
print(f"Data mais antiga: {data_mais_antiga:%Y-%m-%d}")
print(f"Data mais recente: {data_mais_recente:%Y-%m-%d}")
print(f"Dias entre as transações: {dias_periodo}")


===== PERÍODO ANALISADO =====
Data mais antiga: 2026-01-05
Data mais recente: 2026-03-25
Dias entre as transações: 79



## 5. Agrupamento mensal e métricas financeiras

Nesta etapa, as transações válidas serão agrupadas
por mês para calcular as seguintes métricas:

- Quantidade de transações
- Total de créditos
- Total de débitos
- Saldo mensal
- Valor médio das transações
- Maior valor registrado
- Menor valor registrado


In [117]:

def gerar_relatorio(transacoes_validas):

    resumo_mensal = {}

    for transacao in transacoes_validas:

        mes = transacao["data"].strftime("%Y-%m")
        valor = transacao["valor"]
        tipo = transacao["tipo"]

        # Criar o mês caso ainda não exista
        if mes not in resumo_mensal:
            resumo_mensal[mes] = {
                "quantidade": 0,
                "total_credito": 0.0,
                "total_debito": 0.0,
                "soma_valores": 0.0,
                "maior_valor": valor,
                "menor_valor": valor
            }

        dados_mes = resumo_mensal[mes]

        # Contar transações
        dados_mes["quantidade"] += 1

        # Somar os valores
        dados_mes["soma_valores"] += valor

        # Separar créditos e débitos
        if tipo == "credito":
            dados_mes["total_credito"] += valor
        else:
            dados_mes["total_debito"] += valor

        # Identificar maior e menor valor
        dados_mes["maior_valor"] = max(
            dados_mes["maior_valor"], valor
        )

        dados_mes["menor_valor"] = min(
            dados_mes["menor_valor"], valor
        )

    # Calcular as métricas finais
    for mes, dados in resumo_mensal.items():

        dados["saldo"] = round(
            dados["total_credito"] - dados["total_debito"], 2
        )

        dados["media"] = round(
            dados["soma_valores"] / dados["quantidade"], 2
        )

        # Arredondar valores monetários
        for campo in [
            "total_credito",
            "total_debito",
            "maior_valor",
            "menor_valor"
        ]:
            dados[campo] = round(dados[campo], 2)

        del dados["soma_valores"]

    return dict(sorted(resumo_mensal.items()))


In [118]:

resumo_mensal = gerar_relatorio(validas)

print("===== RESUMO FINANCEIRO MENSAL =====")

for mes, dados in resumo_mensal.items():
    print(f"\nMês: {mes}")
    print(f"Transações: {dados['quantidade']}")
    print(f"Créditos: R$ {dados['total_credito']:.2f}")
    print(f"Débitos: R$ {dados['total_debito']:.2f}")
    print(f"Saldo: R$ {dados['saldo']:.2f}")
    print(f"Média: R$ {dados['media']:.2f}")
    print(f"Maior valor: R$ {dados['maior_valor']:.2f}")
    print(f"Menor valor: R$ {dados['menor_valor']:.2f}")


===== RESUMO FINANCEIRO MENSAL =====

Mês: 2026-01
Transações: 5
Créditos: R$ 4700.00
Débitos: R$ 520.40
Saldo: R$ 4179.60
Média: R$ 1044.08
Maior valor: R$ 3500.00
Menor valor: R$ 89.90

Mês: 2026-02
Transações: 5
Créditos: R$ 18500.00
Débitos: R$ 890.00
Saldo: R$ 17610.00
Média: R$ 3878.00
Maior valor: R$ 15000.00
Menor valor: R$ 120.00

Mês: 2026-03
Transações: 5
Créditos: R$ 16000.00
Débitos: R$ 1009.90
Saldo: R$ 14990.10
Média: R$ 3401.98
Maior valor: R$ 12500.00
Menor valor: R$ 99.90



## 6. Identificação de transações suspeitas

Nesta etapa, serão identificadas as transações
financeiras com valores superiores a R$ 10.000,00.

As transações identificadas serão armazenadas
separadamente para inclusão no relatório final.

Uma transação suspeita não será descartada,
pois pode representar uma operação válida.


In [119]:

def identificar_suspeitas(transacoes_validas):

    suspeitas = []

    for transacao in transacoes_validas:

        if transacao["valor"] > LIMITE_SUSPEITO:
            suspeitas.append(transacao)

    return suspeitas


In [120]:

suspeitas = identificar_suspeitas(validas)

print("===== TRANSAÇÕES SUSPEITAS =====")

if suspeitas:
    for transacao in suspeitas:
        print(
            f"ID: {transacao['id']} | "
            f"Cliente: {transacao['cliente_id']} | "
            f"Data: {transacao['data']:%Y-%m-%d} | "
            f"Valor: R$ {transacao['valor']:.2f}"
        )
else:
    print("Nenhuma transação suspeita encontrada.")


===== TRANSAÇÕES SUSPEITAS =====
ID: 7 | Cliente: CLI003 | Data: 2026-02-08 | Valor: R$ 15000.00
ID: 12 | Cliente: CLI002 | Data: 2026-03-05 | Valor: R$ 12500.00



## 7. Exportação do relatório em JSON

Nesta etapa, os resultados da análise financeira
serão organizados e exportados para um arquivo JSON.

O relatório conterá:
- Data de geração
- Quantidade de transações válidas e inválidas
- Resumo financeiro mensal
- Transações potencialmente suspeitas
- Período analisado


In [121]:

def salvar_json(dados, caminho="relatorio.json"):

    try:
        with open(caminho, "w", encoding="utf-8") as arquivo:
            json.dump(
                dados,
                arquivo,
                ensure_ascii=False,
                indent=2
            )

        print(f"Relatório salvo em '{caminho}'.")

    except OSError as erro:
        print(f"Erro ao salvar relatório: {erro}")


In [122]:
relatorio = {
    "gerado_em": datetime.now().strftime("%Y-%m-%d"),

    "total_transacoes_validas": len(validas),

    "total_transacoes_invalidas": len(invalidas),

    "periodo_analisado": {
        "data_inicial": data_mais_antiga.strftime("%Y-%m-%d"),
        "data_final": data_mais_recente.strftime("%Y-%m-%d"),
        "dias": dias_periodo
    },

    "resumo_mensal": resumo_mensal,

    "transacoes_suspeitas": [
        {
            "id": transacao["id"],
            "cliente_id": transacao["cliente_id"],
            "data": transacao["data"].strftime("%Y-%m-%d"),
            "valor": transacao["valor"]
        }
        for transacao in suspeitas
    ]
}


In [123]:
salvar_json(relatorio)

Relatório salvo em 'relatorio.json'.


In [124]:

with open("relatorio.json", "r", encoding="utf-8") as arquivo:
    dados_json = json.load(arquivo)

print("===== VERIFICAÇÃO DO JSON =====")
print("Transações válidas:", dados_json["total_transacoes_validas"])
print("Transações inválidas:", dados_json["total_transacoes_invalidas"])
print("Meses analisados:", list(dados_json["resumo_mensal"].keys()))
print("Transações suspeitas:", len(dados_json["transacoes_suspeitas"]))


===== VERIFICAÇÃO DO JSON =====
Transações válidas: 15
Transações inválidas: 5
Meses analisados: ['2026-01', '2026-02', '2026-03']
Transações suspeitas: 2



## 8. Exibição formatada do relatório

Nesta etapa, será implementada uma função para
apresentar os resultados da análise financeira
de maneira organizada e legível.

O relatório exibirá as métricas mensais,
o período analisado, o resumo da limpeza
e as transações potencialmente suspeitas.


In [125]:

def formatar_moeda(valor):
    valor_formatado = f"{valor:,.2f}"

    valor_formatado = (
        valor_formatado
        .replace(",", "X")
        .replace(".", ",")
        .replace("X", ".")
    )

    return f"R$ {valor_formatado}"


In [126]:

def exibir_relatorio(relatorio):

    print("=" * 45)
    print("       RELATÓRIO FINANCEIRO CLEARBANK")
    print("=" * 45)

    # Informações gerais
    print("\n===== RESUMO DA LIMPEZA =====")
    print(
        "Total de linhas lidas:",
        relatorio["total_transacoes_validas"]
        + relatorio["total_transacoes_invalidas"]
    )
    print("Linhas válidas:", relatorio["total_transacoes_validas"])
    print("Linhas inválidas:", relatorio["total_transacoes_invalidas"])

    # Período analisado
    periodo = relatorio["periodo_analisado"]

    print("\n===== PERÍODO ANALISADO =====")
    print("Data inicial:", periodo["data_inicial"])
    print("Data final:", periodo["data_final"])
    print("Dias entre transações:", periodo["dias"])

    # Relatório mensal
    print("\n===== RELATÓRIO MENSAL =====")

    for mes, dados in relatorio["resumo_mensal"].items():

        print(f"\nMês: {mes}")
        print(f"  Transações:   {dados['quantidade']}")
        print(f"  Total crédito: {formatar_moeda(dados['total_credito'])}")
        print(f"  Total débito:  {formatar_moeda(dados['total_debito'])}")
        print(f"  Saldo:         {formatar_moeda(dados['saldo'])}")
        print(f"  Média:         {formatar_moeda(dados['media'])}")
        print(f"  Maior valor:   {formatar_moeda(dados['maior_valor'])}")
        print(f"  Menor valor:   {formatar_moeda(dados['menor_valor'])}")

    # Transações suspeitas
    print("\n===== TRANSAÇÕES SUSPEITAS =====")

    suspeitas = relatorio["transacoes_suspeitas"]

    if suspeitas:
        for transacao in suspeitas:
            print(
                f"ID: {transacao['id']} | "
                f"Cliente: {transacao['cliente_id']} | "
                f"Data: {transacao['data']} | "
                f"Valor: {formatar_moeda(transacao['valor'])}"
            )
    else:
        print("Nenhuma transação suspeita encontrada.")

    print("\n" + "=" * 45)
    print("          FIM DO RELATÓRIO")
    print("=" * 45)


In [127]:
exibir_relatorio(relatorio)

       RELATÓRIO FINANCEIRO CLEARBANK

===== RESUMO DA LIMPEZA =====
Total de linhas lidas: 20
Linhas válidas: 15
Linhas inválidas: 5

===== PERÍODO ANALISADO =====
Data inicial: 2026-01-05
Data final: 2026-03-25
Dias entre transações: 79

===== RELATÓRIO MENSAL =====

Mês: 2026-01
  Transações:   5
  Total crédito: R$ 4.700,00
  Total débito:  R$ 520,40
  Saldo:         R$ 4.179,60
  Média:         R$ 1.044,08
  Maior valor:   R$ 3.500,00
  Menor valor:   R$ 89,90

Mês: 2026-02
  Transações:   5
  Total crédito: R$ 18.500,00
  Total débito:  R$ 890,00
  Saldo:         R$ 17.610,00
  Média:         R$ 3.878,00
  Maior valor:   R$ 15.000,00
  Menor valor:   R$ 120,00

Mês: 2026-03
  Transações:   5
  Total crédito: R$ 16.000,00
  Total débito:  R$ 1.009,90
  Saldo:         R$ 14.990,10
  Média:         R$ 3.401,98
  Maior valor:   R$ 12.500,00
  Menor valor:   R$ 99,90

===== TRANSAÇÕES SUSPEITAS =====
ID: 7 | Cliente: CLI003 | Data: 2026-02-08 | Valor: R$ 15.000,00
ID: 12 | Cliente: CL


## 9. Execução principal

Esta etapa integra as funções desenvolvidas
anteriormente para realizar o processamento
completo das transações financeiras.

O fluxo contempla leitura, validação, limpeza,
agrupamento mensal, identificação de transações
suspeitas, exibição e exportação do relatório.


In [128]:

def executar_analise():

    # 1. Ler as transações
    transacoes = ler_transacoes("transacoes.csv")

    # 2. Validar os registros
    validas = []
    invalidas = []

    for transacao in transacoes:
        resultado = validar_transacao(transacao)

        if resultado is not None:
            validas.append(resultado)
        else:
            invalidas.append(transacao)

    # Evitar cálculos com lista vazia
    if not validas:
        print("Nenhuma transação válida encontrada.")
        return

    # 3. Identificar o período analisado
    data_inicial = min(t["data"] for t in validas)
    data_final = max(t["data"] for t in validas)

    dias = (data_final - data_inicial).days

    # 4. Gerar o resumo financeiro
    resumo = gerar_relatorio(validas)

    # 5. Identificar transações suspeitas
    suspeitas = identificar_suspeitas(validas)

    # 6. Organizar o relatório
    relatorio = {
        "gerado_em": datetime.now().strftime("%Y-%m-%d"),

        "total_transacoes_validas": len(validas),

        "total_transacoes_invalidas": len(invalidas),

        "periodo_analisado": {
            "data_inicial": data_inicial.strftime("%Y-%m-%d"),
            "data_final": data_final.strftime("%Y-%m-%d"),
            "dias": dias
        },

        "resumo_mensal": resumo,

        "transacoes_suspeitas": [
            {
                "id": t["id"],
                "cliente_id": t["cliente_id"],
                "data": t["data"].strftime("%Y-%m-%d"),
                "valor": t["valor"]
            }
            for t in suspeitas
        ]
    }

    # 7. Exibir e salvar o relatório
    exibir_relatorio(relatorio)
    salvar_json(relatorio)


In [129]:
executar_analise()

       RELATÓRIO FINANCEIRO CLEARBANK

===== RESUMO DA LIMPEZA =====
Total de linhas lidas: 20
Linhas válidas: 15
Linhas inválidas: 5

===== PERÍODO ANALISADO =====
Data inicial: 2026-01-05
Data final: 2026-03-25
Dias entre transações: 79

===== RELATÓRIO MENSAL =====

Mês: 2026-01
  Transações:   5
  Total crédito: R$ 4.700,00
  Total débito:  R$ 520,40
  Saldo:         R$ 4.179,60
  Média:         R$ 1.044,08
  Maior valor:   R$ 3.500,00
  Menor valor:   R$ 89,90

Mês: 2026-02
  Transações:   5
  Total crédito: R$ 18.500,00
  Total débito:  R$ 890,00
  Saldo:         R$ 17.610,00
  Média:         R$ 3.878,00
  Maior valor:   R$ 15.000,00
  Menor valor:   R$ 120,00

Mês: 2026-03
  Transações:   5
  Total crédito: R$ 16.000,00
  Total débito:  R$ 1.009,90
  Saldo:         R$ 14.990,10
  Média:         R$ 3.401,98
  Maior valor:   R$ 12.500,00
  Menor valor:   R$ 99,90

===== TRANSAÇÕES SUSPEITAS =====
ID: 7 | Cliente: CLI003 | Data: 2026-02-08 | Valor: R$ 15.000,00
ID: 12 | Cliente: CL

## 10. Testes de validação e tratamento de erros

Os testes a seguir verificam a rejeição de registros incompletos, o tratamento de arquivos ausentes ou vazios e a validação de valores numéricos inválidos. São executados separadamente da análise principal.

### 10.1. Campos ausentes e valores nulos

In [113]:

# Registro com campo ausente
teste_incompleto = {
    "id": "21",
    "cliente_id": "CLI001",
    "tipo": "credito",
    "valor": "500.00"
}

# Registro com ID nulo
teste_nulo = {
    "id": None,
    "data": "2026-03-10",
    "cliente_id": "CLI001",
    "tipo": "credito",
    "valor": "500.00",
    "descricao": "Teste",
    "categoria": "transferencia"
}

print("Campo ausente:", validar_transacao(teste_incompleto))
print("ID nulo:", validar_transacao(teste_nulo))


Campo ausente: None
ID nulo: None


### 10.2. Arquivo CSV inexistente

In [ ]:
resultado_inexistente = ler_transacoes("arquivo_inexistente.csv")
print("Resultado:", resultado_inexistente)
assert resultado_inexistente == []

### 10.3. Arquivo CSV vazio

O arquivo de teste é criado em uma pasta temporária, que é removida automaticamente após o teste.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as pasta_temporaria:
    arquivo_vazio = Path(pasta_temporaria) / "teste_vazio.csv"
    arquivo_vazio.write_text("", encoding="utf-8")
    resultado_vazio = ler_transacoes(str(arquivo_vazio))
    print("Resultado:", resultado_vazio)
    assert resultado_vazio == []

### 10.4. Valores numéricos inválidos

In [130]:

# Testar valores inválidos
valores_invalidos = ["abc", "-100", "0", "NaN", "inf"]

for valor in valores_invalidos:
    transacao_teste = {
        "id": "21",
        "data": "2026-03-10",
        "cliente_id": "CLI001",
        "tipo": "credito",
        "valor": valor,
        "descricao": "Teste",
        "categoria": "transferencia"
    }

    resultado = validar_transacao(transacao_teste)

    print(f"Valor {valor}: {resultado}")


Valor abc: None
Valor -100: None
Valor 0: None
Valor NaN: None
Valor inf: None
